In [1]:
#  Setup and Imports
import pandas as pd
import pyodbc
from src.utils.db_connector import get_pyodbc_conn_string
print("Setup complete. Functions and connection utilities loaded.")

Setup complete. Functions and connection utilities loaded.


In [3]:
# Reading data from sql
def read_from_bronze(table_names: list) -> dict:
    print("\n--- Reading Data from Bronze Schema into Pandas Memory ---")

    conn_str = get_pyodbc_conn_string()
    cnxn = pyodbc.connect(conn_str)

    dataframes = {}
    for table_name in table_names:
        print(f"Reading table: {table_name}...")
        sql_query = f"SELECT * FROM bronze.{table_name}"
        dataframes[table_name] = pd.read_sql(sql_query, cnxn)
        print(f"Loaded {len(dataframes[table_name]):,} rows from bronze.{table_name}")

    cnxn.close()
    return dataframes

CORE_TABLES = [
    'orders_raw', 'customers_raw', 'order_items_raw', 'reviews_raw',
    'products_raw', 'category_trans_raw', 'payments_raw', 'sellers_raw',
    'geolocation_raw'
]

print("--- Starting Read Operation from SQL Bronze Schema ---")
try:
    # This calls the function you defined in src/etl/etl_functions.py
    dfs = read_from_bronze(CORE_TABLES)
    print("\n✅ Successfully loaded all 9 tables into Pandas DataFrames (dfs dictionary).")
    print("DataFrames loaded: ", dfs.keys())
except Exception as e:
    print(f"❌ FAILED to read from SQL Bronze. Error: {e}")

--- Starting Read Operation from SQL Bronze Schema ---

--- Reading Data from Bronze Schema into Pandas Memory ---


C:\Users\Ayush\AppData\Local\Temp\ipykernel_19260\1544798567.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataframes[table_name] = pd.read_sql(sql_query, cnxn)


Reading table: orders_raw...
Loaded 99,441 rows from bronze.orders_raw
Reading table: customers_raw...
Loaded 99,441 rows from bronze.customers_raw
Reading table: order_items_raw...
Loaded 225,300 rows from bronze.order_items_raw
Reading table: reviews_raw...
Loaded 198,448 rows from bronze.reviews_raw
Reading table: products_raw...
Loaded 65,902 rows from bronze.products_raw
Reading table: category_trans_raw...
Loaded 142 rows from bronze.category_trans_raw
Reading table: payments_raw...
Loaded 207,772 rows from bronze.payments_raw
Reading table: sellers_raw...
Loaded 6,190 rows from bronze.sellers_raw
Reading table: geolocation_raw...
Loaded 2,000,326 rows from bronze.geolocation_raw

✅ Successfully loaded all 9 tables into Pandas DataFrames (dfs dictionary).
DataFrames loaded:  dict_keys(['orders_raw', 'customers_raw', 'order_items_raw', 'reviews_raw', 'products_raw', 'category_trans_raw', 'payments_raw', 'sellers_raw', 'geolocation_raw'])


In [10]:
# orders Raw cleaning
orders_raw_df=dfs['orders_raw']
orders_raw_df.head()
orders_raw_df.isnull().sum()
def check_timestamp_errors(df: pd.DataFrame) -> pd.Series:
    purchase_after_approved = (df['order_purchase_timestamp'] > df['order_approved_at'])

    approved_after_carrier = (df['order_approved_at'] > df['order_delivered_carrier_date'])

    carrier_after_customer = (df['order_delivered_carrier_date'] > df['order_delivered_customer_date'])

    error_mask = (purchase_after_approved.fillna(False)) | \
                 (approved_after_carrier.fillna(False)) | \
                 (carrier_after_customer.fillna(False))

    return error_mask
error_rows=check_timestamp_errors(orders_raw_df)
total_errors=error_rows.sum()
print("Total logical timestamp error found : ", total_errors)

def clean_timestamp_data(df: pd.DataFrame, error_mask: pd.Series)->pd.DataFrame:
    df_cleaned=df[~error_mask]
    deleted_count=error_mask.sum()
    print(f"Total logical errors dropped: {deleted_count:,}")
    print(f"Final Row Count: {len(df_cleaned):,}")
    return df_cleaned
orders_raw_df=clean_timestamp_data(orders_raw_df,error_rows)

Total logical timestamp error found :  1382
Total logical errors dropped: 1,382
Final Row Count: 98,059


In [11]:
# customers Raw cleaning
customer_raw_df=dfs['customers_raw']
customer_raw_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
2,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG
3,9fb35e4ed6f0a14a4977cd9aea4042bb,2a7745e1ed516b289ed9b29c7d0539a5,39400,montes claros,MG
4,1f1c7bf1c9b041b292af6c1c4470b753,3151a81801c8386361b62277d7fa5ecf,95110,caxias do sul,RS
